In [ ]:
from essential.kegg_modules import list_kegg_modules, kegg_module_to_graph, metabolic_to_operational_graph
from essential.plot_pathways import plot_pathway_results, plot_metabolic_pathway
from essential.utils import PLOTNINE_DEFAULT_THEME_2
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotnine as gg
import os
from tqdm import tqdm
import scanpy as sc

from essential.data import load_fitness_data
from essential.pathway_discontinuity import PathwayDiscontinuity


plt.rcParams["svg.fonttype"] = "none"

def plot_umap_genes(adata, genes):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(genes)]
    obs_subset["target"] = obs_subset["target"].astype(str)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point()
        + gg.geom_point(obs_subset, gg.aes(color="target"), size=2)
        + gg.theme_minimal()
    )
    return fig

def plot_umap_equiv_classes(adata, gene_to_class, class_color_mapping, plot_legend=True, point_size=1.5):
    obs_subset = adata.obs.loc[lambda x: x["target"].isin(gene_to_class.keys())]
    obs_subset["equivalence_class"] = obs_subset["target"].map(gene_to_class).sample(frac=1)

    fig = (
        gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
        + gg.geom_point(size=0.7, stroke=0)
        + gg.geom_point(obs_subset, gg.aes(color="equivalence_class"), size=point_size, stroke=0.0)
        + gg.scale_color_manual(values=class_color_mapping)
        + gg.theme_minimal()
    )
    if not plot_legend:
        fig = fig + gg.theme(legend_position="none")
    return fig


def plot_expression_data(adata, perturbations, gene_names):
    adata_selected = adata[adata.obs["target"].isin(perturbations)]
    # sc.pl.heatmap(adata_selected, var_names=gene_names, groupby="target", swap_axes=False)
    dp = sc.pl.dotplot(adata_selected, var_names=gene_names, groupby="target", swap_axes=False, show=False)                                                                
    ax = dp["mainplot_ax"]                                                                                                                                                 
    ax.set_xlabel("measurement")                                                                                                                                           
    ax.set_ylabel("perturbation")                                                                                                                                            

def compare_pairs(adata, gene1, gene2):
    adata_pair = adata[adata.obs["target"].isin([gene1, gene2])]
    sc.tl.rank_genes_groups(adata_pair, "target", method="wilcoxon")
    sc.pl.rank_genes_groups(adata_pair, n_genes=20, show=False)
    de_genes = sc.get.rank_genes_groups_df(adata_pair, group=gene1)

    for _, row in de_genes.sort_values("pvals").head(20).iterrows():
        print(f"{row['names']} pval:{row['pvals_adj']:0.2e}, log2FC:{row['logfoldchanges']:0.2f}")
    return de_genes

def rename_nodes(g, renamer):
    G_copy = g.copy()
    for node, new_name in renamer.items():
        if node in G_copy.nodes:
            G_copy.nodes[node]["name"] = new_name
    return G_copy


def plot_fitness_data(fitness_data, plot_info):
    gene_names = plot_info["gene_color_mapping"].keys()
    fitness_data_ = fitness_data.loc[lambda x: x["gene"].isin(gene_names)]
    fitness_data_["gene"] = pd.Categorical(fitness_data_["gene"], categories=gene_names, ordered=True)

    fig = (
        gg.ggplot(fitness_data_, gg.aes(x="gene", y="T3", fill="gene"))
        + gg.scale_fill_manual(plot_info["gene_color_mapping"])
        + gg.geom_col()
        + PLOTNINE_DEFAULT_THEME_2
        + gg.theme(
            legend_position="none"
        )
        + gg.coord_flip()
        + gg.labs(
            y="fitness (T+24h), (Calvo-Villamanan et al., 2020)", x=""
        )
    )
    return fig

In [ ]:
fitness_data_path = "../../data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
# fitness_data_path = "../../../../data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
fitness_data = load_fitness_data(fitness_data_path).groupby("gene")[["T1", "T2", "T3", "T4"]].mean().reset_index()


adata_path = "/workspace/data/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed.h5ad"
adata = sc.read_h5ad(adata_path)